# 02_feature_extraction.ipynb
Generación de embeddings (features) a partir del dataset preprocesado.
- Lee los datos desde `repo-root/data/desayuno_preprocessed/`
- Extrae embeddings con un backbone pre-entrenado (ResNet50 por defecto)
- Guarda el resultado en `repo-root/backend/models/embeddings.pkl`
- Usa GPU si está disponible


# 🟩 Celda 1 — Imports y rutas de trabajo

In [1]:
import os
import pandas as pd
import numpy as np
import pickle

# 🔧 Configuración de rutas
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data", "desayuno_preprocessed")
MODELS_DIR = os.path.join(BASE_DIR, "backend", "models")
RESULTS_DIR = os.path.join(BASE_DIR, "backend", "results")

# Crear carpetas si no existen
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("📂 Rutas configuradas correctamente:")
print("DATA_DIR:", DATA_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("RESULTS_DIR:", RESULTS_DIR)


📂 Rutas configuradas correctamente:
DATA_DIR: c:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\data\desayuno_preprocessed
MODELS_DIR: c:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\backend\models
RESULTS_DIR: c:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\backend\results


# 🟩 Celda 2 — Cargar el dataset preprocesado

🔹 En esta celda asumimos que tu compañera te pasó un CSV limpio dentro del zip.
Si tiene otro nombre, cámbialo en la variable DATA_FILE.

In [3]:
# 🧾 Nombre del archivo CSV preprocesado (ajusta si el nombre cambia)
DATA_FILE = "dataset_desayuno_summary.csv"
DATA_PATH = os.path.join(DATA_DIR, DATA_FILE)

# Cargar dataset
df = pd.read_csv(DATA_PATH)

print("✅ Dataset cargado correctamente.")
print("🔹 Forma del dataset:", df.shape)
df.head()


✅ Dataset cargado correctamente.
🔹 Forma del dataset: (21, 2)


,clase_nombre,num_imagenes
0,apple_pie,998
1,beignets,997
2,bread_pudding,996
3,breakfast_burrito,999
4,cannoli,998


# 🟩 Celda 3 — Análisis inicial rápido (EDA resumido)

In [4]:
# 📊 Información general
print("🔍 Información del dataset:\n")
print(df.info())

print("\n📈 Estadísticas descriptivas:")
display(df.describe())

# Ver distribución de la variable objetivo (si existe)
if 'target' in df.columns:
    print("\n🎯 Distribución de clases:")
    print(df['target'].value_counts(normalize=True) * 100)
else:
    print("⚠️ No se encontró columna 'target'. Asegúrate de definirla antes de continuar.")


🔍 Información del dataset:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   clase_nombre  21 non-null     object
 1   num_imagenes  21 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 468.0+ bytes
None

📈 Estadísticas descriptivas:


,num_imagenes
count,21.000000
mean,999.047619
std,1.160870
min,996.000000
25%,998.000000
50%,999.000000
75%,1000.000000
max,1000.000000


⚠️ No se encontró columna 'target'. Asegúrate de definirla antes de continuar.


# 🟩 Celda 4 — Separar características y etiquetas

🔹 Aquí asumimos que la columna objetivo se llama target.

In [5]:
TARGET_COL = "target"  # cambia si tiene otro nombre

# Separar X e y
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print("✅ Datos separados correctamente.")
print("X shape:", X.shape)
print("y shape:", y.shape)


KeyError: "['target'] not found in axis"

# 🟩 Celda 5 — Generar embeddings o codificación numérica

Aquí transformaremos variables categóricas en numéricas.
Si ya tienes embeddings (por ejemplo, word embeddings, CNN features, etc.), reemplazaremos esta parte más adelante.
Por ahora hacemos una versión segura y eficiente para tabulares.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Detectamos tipos de columnas
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("🧩 Columnas categóricas:", cat_cols)
print("🔢 Columnas numéricas:", num_cols)

# Definir transformador
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ]
)

# Aplicar transformación
X_embedded = preprocessor.fit_transform(X)

print("✅ Embeddings generados correctamente.")
print("🔹 Nueva forma de X:", X_embedded.shape)


# 🟩 Celda 6 — Guardar embeddings (en /models)

In [ ]:
X_pkl_path = os.path.join(MODELS_DIR, "X_embeddings.pkl")
y_pkl_path = os.path.join(MODELS_DIR, "y_labels.pkl")

with open(X_pkl_path, "wb") as f:
    pickle.dump(X_embedded, f)

with open(y_pkl_path, "wb") as f:
    pickle.dump(y, f)

print("💾 Embeddings guardados en:")
print("-", X_pkl_path)
print("-", y_pkl_path)


# 🟩 Celda 7 — Guardar resumen del EDA en /results

In [ ]:
summary_path = os.path.join(RESULTS_DIR, "feature_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("📊 RESUMEN DE FEATURES\n")
    f.write("=====================\n\n")
    f.write(f"Total de filas: {df.shape[0]}\n")
    f.write(f"Total de columnas: {df.shape[1]}\n\n")
    f.write("Columnas numéricas:\n")
    f.write(", ".join(num_cols) + "\n\n")
    f.write("Columnas categóricas:\n")
    f.write(", ".join(cat_cols) + "\n\n")
    f.write(f"Forma final de embeddings: {X_embedded.shape}\n")

print(f"✅ Resumen guardado en {summary_path}")
